# Notebook 2: HMM Library 
-  **from-scratch Gaussian HMM** (forward-backward, Baum-Welch, Viterbi)
-  **from-scratch Gaussian Naive Bayes baseline**
-  **pipeline helpers** that train and evaluate one feature source

### HMM
### setup
- ~140 patients in OASIS-2, with 2 to 5 clinic visits
- at every visit we measure clinical numbers, and each visit has a CDR label: 0 = healthy, 0.5 = very mild, 1 = mild dementia
- goal: predict CDR + understand how patients move between CDR levels over time

- there's a **hidden severity** at each visit we can't measure directly
- each hidden state produces noisy clinical numbers when we observe it
- the next hidden state depends only on the current one (the **Markov assumption**)
- 3 hidden states, hopefully matching CDR 0, 0.5, 1

### The 3 things the model learns
- **start probabilities**: how likely each hidden state is at the very first visit
- **transition probabilities**: 3x3 table of "if I'm in state X now, what's the chance I'm in state Y at  next visit"
- **emissions**: per state Gaussian over the clinical features

### How we train it (Baum-Welch loop)
1. start with a guess for all the parameters (warm-started from per-CDR averages)
2. **forward-backward**: for every visit, compute "how confident are we this visit was in each hidden state?"
3. **update step**: use those confidences to recompute starts, transitions, and Gaussians
4. check progress, repeat if it improved
5. Baum-Welch can get stuck, so we run it from **8 different starting points** and keep the best

### How we use it on a new patient (Viterbi)
- Viterbi finds the **most likely sequence of hidden states** for one patient

### The naming problem (Hungarian algorithm)
- after training, hidden states are just numbered 0/1/2 with no meaning
- the Hungarian algorithm picks the one-to-one `state -> CDR` mapping that gets the most training visits right

#### Imports

In [160]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA
from sklearn.metrics import (balanced_accuracy_score, classification_report, confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (7, 4)
CDR_VALUES = np.array([0.0, 0.5, 1.0])

def cdr_to_class_idx(cdr_values):
    """cdr {0, 0.5, 1} to int class {0, 1, 2}"""
    return np.round(np.asarray(cdr_values, dtype=float) * 2).astype(int)

def log_sum_exp(values, axis=None):
    """numerically stable log(sum(exp(values))), used in the HMM forward-backward"""
    # subtract the max before exponentiating so we don't overflow on big values
    max_val = np.max(values, axis=axis, keepdims=True)
    return np.squeeze(max_val, axis=axis) + np.log(np.sum(np.exp(values - max_val), axis=axis))


#### Piece 1: class

Holds 4 things:
- `startprob_`: start probabilities (length 3)
- `transmat_`: 3x3 **transition matrix** (every row sums to 1)
- `means_`: 3 x n_features,  mean of each state's Gaussian
- `covars_`: 3 x n_features, variance of each state's Gaussian (**diagonal covariance**: each feature has its own variance, no cross-feature correlations)

We set them outside before calling `fit` (warm start), which runs Baum-Welch and updates them.

In [161]:
class GaussianHMMScratch:
    """3-state Gaussian HMM with diagonal covariance, written from scratch."""
    def __init__(self, n_states=3, n_iter=600, tol=1e-5, random_state=42):
        self.n_states = n_states
        # max baum-welch rounds before we give up
        self.n_iter = n_iter
        # stop when the log-likelihood improves by less than this between rounds
        self.tol = tol
        self.random_state = random_state
        # set externally before fit (warm start)
        self.startprob_ = None
        self.transmat_  = None
        self.means_     = None
        self.covars_    = None


### Piece 2: emission probabilities

- **emission** is the per-state observation distribution
- ours is a Gaussian: each state has a mean and variance for every feature, so the emission is "how close is this visit to that state's average"
- we work in **log space** because raw probabilities can be tiny and would underflow to zero in floating point. log space turns multiplications into additions and keeps numbers in a sane range

- `log_probs[t][k]` = log P(observation at visit t | hidden state k), under a diagonal-covariance Gaussian

In [162]:
def _log_emission_probs(self, X):
    """log P(observation | state) for every (visit, state) pair, diagonal-covariance gaussian"""
    n_states, n_features = self.means_.shape
    seq_len = X.shape[0]
    log_two_pi = np.log(2.0 * np.pi)
    log_probs = np.empty((seq_len, n_states))
    for state in range(n_states):
        # squared distance from this state's average, scaled by variance
        diff = X - self.means_[state]
        variance = self.covars_[state]
        log_probs[:, state] = -0.5 * (n_features * log_two_pi + np.log(variance).sum() + ((diff * diff) / variance).sum(axis=1))
    return log_probs

GaussianHMMScratch._log_emission_probs = _log_emission_probs


### Piece 3: forward-backward

For one patient sequence compute:
- `log_state_probs[t][k]`: how confident the model is that visit `t` was in hidden state `k`, given the **whole** sequence
- `log_pair_probs[t][i][j]`: how confident it is that visit `t` was in state `i` AND visit `t+1` was in state `j`

two passes:
- **forward**: walk visits left to right, accumulating "how likely is the sequence so far AND ending in each state"
- **backward**: walk right to left, accumulating "how likely is the rest given we're in each state now"
- combine them and divide by the total to get per-visit state probabilities

- `log_forward[t][k]` = log P(observations 0..t AND ending in state k at visit t)
- `log_backward[t][k]` = log P(observations t+1..end | being in state k at visit t)
- `log_state_probs[t][k]` = `log_forward[t][k]` + `log_backward[t][k]` − `log_total`
- `log_pair_probs[t][i][j]` = log P(visit t in state i AND visit t+1 in state j | full sequence)

In [ ]:
def _forward_backward(self, log_emission_probs):
    """Forward-backward on ONE patient sequence. Returns log_state_probs, log_pair_probs, log_total."""
    seq_len, n_states = log_emission_probs.shape
    log_start = np.log(self.startprob_ + 1e-300)
    log_trans = np.log(self.transmat_ + 1e-300)

    # forward pass
    log_forward = np.empty((seq_len, n_states))
    # first visit has no previous state
    log_forward[0] = log_start + log_emission_probs[0]
    for t in range(1, seq_len):
        # sum over all possible previous states (in log space)
        log_forward[t] = log_emission_probs[t] + log_sum_exp(log_forward[t - 1, :, None] + log_trans, axis=0)

    # backward pass
    log_backward = np.empty((seq_len, n_states))
    # last visit has no future
    log_backward[-1] = 0.0
    for t in range(seq_len - 2, -1, -1):
        # sum over all possible next states
        log_backward[t] = log_sum_exp(log_trans + log_emission_probs[t + 1, None, :] + log_backward[t + 1, None, :], axis=1)

    # total sequence log-likelihood
    log_total = log_sum_exp(log_forward[-1])
    # per-visit state probabilities
    log_state_probs = log_forward + log_backward - log_total

    # joint pair probabilities 
    if seq_len > 1:
        log_pair_probs = np.empty((seq_len - 1, n_states, n_states))
        for t in range(seq_len - 1):
            log_pair_probs[t] = (log_forward[t, :, None] + log_trans + log_emission_probs[t + 1, None, :] + log_backward[t + 1, None, :] - log_total)
    else:
        log_pair_probs = np.zeros((0, n_states, n_states))
    return log_state_probs, log_pair_probs, log_total

GaussianHMMScratch._forward_backward = _forward_backward


### Piece 4: fitting loop (Baum-Welch)

Each round:
1. run forward-backward on every patient to get **soft counts** ( fractional count weighted by how confident we are that a visit was in a particular state). 
2. use the soft counts to recompute  start probs, transition table, means, and variances. 
3. check if the total log likelihood improved. if yes, repeat. if not, stop.


In [164]:
def fit(self, X, lengths):
    """Baum-Welch over multiple patient sequences. 
    X stacks all patients' visits; lengths says where each patient's chunk starts and ends."""
    n_states = self.n_states
    n_features = X.shape[1]
    prev_total = -np.inf

    for iteration in range(self.n_iter):
        # One per parameter, will update
        sum_start_probs = np.zeros(n_states)
        sum_pair_probs  = np.zeros((n_states, n_states))
        sum_state_probs = np.zeros(n_states)
        sum_state_x     = np.zeros((n_states, n_features))
        sum_state_xx    = np.zeros((n_states, n_features))
        total_log_lik   = 0.0

        # Run forward-backward on every patient and get soft counts
        offset = 0
        for length in lengths:
            patient_seq = X[offset:offset + length]
            offset += length
            log_emit = self._log_emission_probs(patient_seq)
            log_state_probs, log_pair_probs, log_lik = self._forward_backward(log_emit)
            # convert to probabilities to accumulate weighted sums
            state_probs = np.exp(log_state_probs)
            pair_probs  = np.exp(log_pair_probs)
            # only  first visit contributes to start distribution
            sum_start_probs += state_probs[0]
            if pair_probs.shape[0] > 0:
                sum_pair_probs += pair_probs.sum(axis=0)
            sum_state_probs += state_probs.sum(axis=0)
            # weighted sums for new means and variances
            sum_state_x  += state_probs.T @ patient_seq
            sum_state_xx += state_probs.T @ (patient_seq * patient_seq)
            total_log_lik += log_lik

        # new start distribution: how often each state appears at visit 0 across patients
        self.startprob_ = sum_start_probs / max(sum_start_probs.sum(), 1e-300)
        # new transition table: each row "starting from state i, where does it go next on average"
        row_totals = sum_pair_probs.sum(axis=1, keepdims=True)
        row_totals = np.where(row_totals < 1e-300, 1.0, row_totals)
        self.transmat_ = sum_pair_probs / row_totals
        # new means: weighted average of observations, weighted by state probability
        self.means_ = sum_state_x / sum_state_probs[:, None]
        # new variances: standard variance formula, weighted
        new_variance = sum_state_xx / sum_state_probs[:, None] - self.means_ ** 2
        self.covars_ = np.clip(new_variance, 1e-3, None)

        # stop if the model fit isn't really improving anymore
        if abs(total_log_lik - prev_total) < self.tol:
            break
        prev_total = total_log_lik

    self.train_log_lik_ = total_log_lik
    return self

GaussianHMMScratch.fit = fit


### Piece 5: score (sequence likelihood)

- returns the total log-likelihood of all the patient sequences under the current parameters
- used to compare random restarts (the one with the highest score wins)
- forward pass only, no backward needed

In [165]:
def score(self, X, lengths):
    """Total log-likelihood of all patient sequences. Forward pass only (no backward needed)."""
    log_start = np.log(self.startprob_ + 1e-300)
    log_trans = np.log(self.transmat_  + 1e-300)
    total = 0.0
    offset = 0
    for length in lengths:
        patient_seq = X[offset:offset + length]
        offset += length
        # forward pass for this patient
        log_emit = self._log_emission_probs(patient_seq)
        seq_len, n_states = log_emit.shape
        log_forward = np.empty((seq_len, n_states))
        log_forward[0] = log_start + log_emit[0]
        for t in range(1, seq_len):
            log_forward[t] = log_emit[t] + log_sum_exp(log_forward[t - 1, :, None] + log_trans, axis=0)
        # the sum at the last visit is the log probability of this whole patient sequence
        total += float(log_sum_exp(log_forward[-1]))
    return total

GaussianHMMScratch.score = score


### Piece 6: Viterbi (decoding a new patient)

- Viterbi picks the **single most likely** sequence of hidden states for one patient (not a distribution)
- forward sweep: at each visit and each state, store the best score and a backpointer to which previous state gave it
- at the end, pick the best final state and walk the backpointers back to recover the path

- `viterbi_log[t][k]` = best score for any path ending at state k at visit t
- `backpointer[t][k]` = which prev state gave that best score

In [166]:
def predict(self, X):
    """Viterbi decoding: most likely state sequence for ONE patient."""
    log_emit = self._log_emission_probs(X)
    seq_len, n_states = log_emit.shape
    log_start = np.log(self.startprob_ +1e-300)
    log_trans = np.log(self.transmat_ +1e-300)

    viterbi_log = np.empty((seq_len, n_states))
    backpointer = np.zeros((seq_len, n_states), dtype=int)
    # first visit has no previous state
    viterbi_log[0] = log_start + log_emit[0]

    # forward sweep: best previous state for each (visit, state)
    for t in range(1, seq_len):
        scores = viterbi_log[t - 1, :, None] + log_trans
        backpointer[t] = np.argmax(scores, axis=0)
        viterbi_log[t] = log_emit[t] + scores[backpointer[t], np.arange(n_states)]

    # backtrack from the best final state
    states = np.empty(seq_len, dtype=int)
    states[-1] = int(np.argmax(viterbi_log[-1]))
    for t in range(seq_len - 2, -1, -1):
        states[t] = backpointer[t + 1, states[t + 1]]
    return states

GaussianHMMScratch.predict = predict


## Gaussian Naive Bayes baseline 
- a  classifier with **no concept of sequence**
- for each CDR class, fit a Gaussian per feature using only that class's training visits
- to predict, ask "which class's Gaussians fit this visit best?"
- "naive" = pretends features are independent of each other

In [ ]:
class GaussianNBScratch:
    """Gaussian Naive Bayes from scratch. Used as a static (non-sequence) baseline."""

    def fit(self, X, y):
        """Fit a per-class mean and variance for every feature."""
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        self.means_ = np.zeros((n_classes, n_features))
        self.variances_ = np.zeros((n_classes, n_features))
        self.log_priors_ = np.zeros(n_classes)
        for i, cls in enumerate(self.classes_):
            mask = (y == cls)
            self.means_[i] = X[mask].mean(axis=0)
            # floor variance for numerical safety
            self.variances_[i] = np.clip(X[mask].var(axis=0), 1e-6, None)
            # log P(class) = log fraction of visits in this class
            self.log_priors_[i] = np.log(mask.mean() + 1e-12)
        return self

    def predict(self, X):
        """Pick the class with the highest log P(class) + log P(visit | class)."""
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        log_two_pi = np.log(2.0 * np.pi)
        scores = np.empty((len(X), n_classes))
        for i in range(n_classes):
            diff = X - self.means_[i]
            variance = self.variances_[i]
            # log prior plus log of the gaussian pdf
            scores[:, i] = (self.log_priors_[i] - 0.5 * (n_features * log_two_pi + np.log(variance).sum() + ((diff * diff) / variance).sum(axis=1)))
        return self.classes_[np.argmax(scores, axis=1)]

## Pipeline helpers
1. **HMM training**: warm start, single restart, multi-restart loop
2. **decoding + alignment**: Viterbi over multiple sequences, Hungarian state-to-CDR map
3. **data wrangling**: pack patient sequences, count CDR transitions, smooth the transition matrix
4. **the orchestrator**: `run_hmm_pipeline` ties it all together

###  part 1: HMM training (warm start + restarts)

`init_clinical_transition_prior` builds an initial transition matrix biased toward " move forward more than backward". Baum-Welch can move away from it during training if the data disagrees.

In [ ]:
def init_clinical_transition_prior(stay_prob=0.68, progress_prob=0.22):
    """Initial transition matrix biased toward sticky states and forward progression."""
    backward_prob = max(0.0, 1.0 - stay_prob - progress_prob)
    A = np.array([
        [stay_prob, progress_prob, backward_prob],
        [backward_prob * 0.5,  stay_prob,             progress_prob + backward_prob * 0.5],
        [backward_prob * 0.5,  backward_prob * 0.5,   stay_prob + progress_prob],
    ], float)
    # normalize each row to sum to 1 
    return A / A.sum(axis=1, keepdims=True)


def fit_hmm_one_restart(X, y, lengths, seed):
    """Warm-start emissions from per-CDR training averages,then run Baum-Welch."""
    rng = np.random.default_rng(seed)
    class_idx = cdr_to_class_idx(y)
    n_states, n_features = 3, X.shape[1]
    means = np.zeros((n_states, n_features))
    variances = np.zeros((n_states, n_features))
    for state in range(n_states):
        in_class = class_idx == state
        # if a class has fewer than 2 visits in this fold, fall back to overall stats
        if in_class.sum() < 2:
            means[state]     = X.mean(axis=0)
            variances[state] = np.clip(X.var(axis=0), 1e-3, None)
        else:
            means[state]     = X[in_class].mean(axis=0)
            variances[state] = np.clip(X[in_class].var(axis=0), 1e-3, None)
    # tiny nudge so each restart starts in a slightly different place
    means += rng.normal(0, 0.04, means.shape)
    hmm_model = GaussianHMMScratch(n_states=n_states, n_iter=600, tol=1e-5, random_state=int(rng.integers(1 << 30)))
    hmm_model.startprob_ = np.full(n_states, 1.0 / n_states)
    hmm_model.transmat_  = init_clinical_transition_prior()
    hmm_model.means_     = means
    hmm_model.covars_    = variances
    hmm_model.fit(X, lengths)
    return hmm_model

def fit_hmm_with_restarts(X, y, lengths, n_restarts=8, base_seed=42):
    """Run Baum-Welch from n_restarts starting points, keep the best by training log-lik."""
    best_log_lik = -np.inf
    best_model = None
    for restart_idx in range(n_restarts):
        # 997 is just a geekforgeeks recommended number so restarts land in very different places
        candidate_model = fit_hmm_one_restart(X, y, lengths, seed=base_seed + 997 * restart_idx)
        candidate_lik = candidate_model.score(X, lengths)
        if candidate_lik > best_log_lik:
            best_log_lik = candidate_lik
            best_model = candidate_model
    return best_model, best_log_lik


###  part 2: decoding/ alignment
- `predict_state_sequence`: Viterbi on every patient
- `hungarian_state_to_cdr_map`: builds a 3x3 match-count table from training visits and lets hungarian pick the best one-to-one `state -> CDR` mapping

In [169]:
def predict_state_sequence(hmm_model, X, lengths):
    """Run Viterbi on every patient and combine the predicted state sequences."""
    all_states = []
    offset = 0
    for length in lengths:
        all_states.append(hmm_model.predict(X[offset:offset + length]))
        offset += length
    return np.concatenate(all_states)

def hungarian_state_to_cdr_map(predicted_states, true_cdr):
    """Pick the one-to-one state -> CDR mapping that maximizes training agreement."""
    # rows = predicted hidden state, cols = true cdr class, value = how often they aligned
    match_counts = np.zeros((3, 3))
    for state, true_class in zip(predicted_states, cdr_to_class_idx(true_cdr)):
        match_counts[state, true_class] += 1
    # scipy minimizes cost, so negate to turn it into "maximize agreement"
    rows, cols = linear_sum_assignment(-match_counts)
    state_to_cdr = {int(rows[i]): float(CDR_VALUES[cols[i]]) for i in range(3)}
    train_agreement = float(match_counts[rows, cols].sum()) / max(len(true_cdr), 1)
    return state_to_cdr, train_agreement


###  part 3: data 

- `pack_patient_sequences`: stacks each patient's visits into one array and a `lengths` list 
- `count_observed_cdr_transitions`: counts CDR-to-CDR transitions in the training set, used for the heatmap
- `smooth_transition_matrix`: adds **Dirichlet prior** before normalizing

Note on Dirichlet prior: a Bayesian smoother. Add small "fake counts" to real counts before normalizing so even sparse rows end up with sensible probabilities.

In [ ]:
def pack_patient_sequences(sequence_df, feature_cols, scaler, patient_ids):
    """Stack each patient's visits into one array, plus a lengths list"""
    feature_chunks, label_chunks, lengths = [], [], []
    for patient_id, group in sequence_df.groupby("Subject ID"):
        # need at least 2 visits for the HMM to see a transition
        if len(group) < 2 or patient_id not in patient_ids:
            continue
        feature_chunks.append(scaler.transform(group[feature_cols].to_numpy(float)))
        label_chunks.append(group["CDR"].to_numpy(float))
        lengths.append(len(group))
    if not feature_chunks:
        return np.empty((0, len(feature_cols))), np.array([]), np.array([], dtype=int)
    return np.vstack(feature_chunks), np.concatenate(label_chunks), np.array(lengths, dtype=int)


def count_observed_cdr_transitions(sequence_df, patient_ids):
    """Count CDR-to-CDR transitions between consecutive visits in the given patient set."""
    counts = np.zeros((3, 3))
    for patient_id, group in sequence_df.groupby("Subject ID"):
        if patient_id not in patient_ids or len(group) < 2:
            continue
        idx_seq = cdr_to_class_idx(group.sort_values("Visit")["CDR"].to_numpy())
        for current_idx, next_idx in zip(idx_seq[:-1], idx_seq[1:]):
            counts[current_idx, next_idx] += 1
    return counts

def smooth_transition_matrix(observed_counts, dirichlet_prior):
    """Add Dirichlet fake counts to the observed counts and renormalize each row."""
    posterior_counts = observed_counts + dirichlet_prior
    return posterior_counts / posterior_counts.sum(axis=1, keepdims=True)


### Helpers part 4: the pipeline

Note: PCA rotates the features into new axes that are uncorrelated. Whitening then rescales each new axis to variance 1. The combo makes the inputs match what our HMM assumes (uncorrelated features, similar scales).

In [171]:
def run_hmm_pipeline(name, sequence_df, feature_cols, train_patient_ids, test_patient_ids, n_pca_components=8, show=True):
    """End-to-end HMM pipeline on one feature set. Returns a metrics dict."""
    # standardize on training rows only so test stats don't leak in
    train_rows = sequence_df["Subject ID"].isin(train_patient_ids)
    feature_scaler = StandardScaler().fit(sequence_df.loc[train_rows, feature_cols].to_numpy(float))
    X_train, y_train, train_lengths = pack_patient_sequences(sequence_df, feature_cols, feature_scaler, train_patient_ids)
    X_test,  y_test,  test_lengths  = pack_patient_sequences(sequence_df, feature_cols, feature_scaler, test_patient_ids)

    # PCA-whiten so the diagonal-gaussian emission assumption holds
    n_components = min(n_pca_components, X_train.shape[1])
    pca = PCA(n_components=n_components, whiten=True, random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca  = pca.transform(X_test)

    # train the HMM with random restarts
    best_model, best_log_lik = fit_hmm_with_restarts(X_train_pca, y_train, train_lengths)

    # match hidden states to cdr labels using training visits
    train_state_seq = predict_state_sequence(best_model, X_train_pca, train_lengths)
    state_to_cdr, train_agreement = hungarian_state_to_cdr_map(train_state_seq, y_train)

    # decode the test set with viterbi and translate back to cdr
    test_state_seq = predict_state_sequence(best_model, X_test_pca, test_lengths)
    y_test_pred = np.array([state_to_cdr[state] for state in test_state_seq], float)
    y_test_idx = cdr_to_class_idx(y_test)
    y_pred_idx = cdr_to_class_idx(y_test_pred)

    confusion = confusion_matrix(y_test_idx, y_pred_idx, labels=[0, 1, 2])
    hmm_macro_f1 = float(f1_score(y_test_idx, y_pred_idx, labels=[0, 1, 2], average="macro", zero_division=0.0))
    hmm_balanced = float(balanced_accuracy_score(y_test_idx, y_pred_idx))

    # majority baseline
    most_common_class = int(np.bincount(cdr_to_class_idx(y_train), minlength=3).argmax())
    y_pred_majority = np.full_like(y_test_idx, most_common_class)
    majority_macro_f1 = float(f1_score(y_test_idx, y_pred_majority, labels=[0, 1, 2], average="macro", zero_division=0.0))
    majority_balanced = float(balanced_accuracy_score(y_test_idx, y_pred_majority))

    # naive bayes baseline on the same pca features
    naive_bayes_model = GaussianNBScratch().fit(X_train_pca, cdr_to_class_idx(y_train))
    y_pred_nb = naive_bayes_model.predict(X_test_pca)
    nb_macro_f1 = float(f1_score(y_test_idx, y_pred_nb, labels=[0, 1, 2], average="macro", zero_division=0.0))
    nb_balanced = float(balanced_accuracy_score(y_test_idx, y_pred_nb))

    if show:
        print(f"{name}")
        print(f"features: {len(feature_cols)} columns | train/test visits: {len(y_train)} / {len(y_test)}")
        print(f"best training log-likelihood: {best_log_lik:.2f}")
        print(f"state -> cdr (hungarian): {state_to_cdr}  (train agreement {train_agreement:.1%})")
        print()
        print(classification_report(y_test_idx, y_pred_idx, labels=[0, 1, 2], target_names=["CDR 0", "CDR 0.5", "CDR 1"], digits=3, zero_division=0.0))
        print(f"  {'majority class':22}  macro f1 = {majority_macro_f1:.3f}   balanced acc = {majority_balanced:.3f}")
        print(f"  {'gaussian naive bayes':22}  macro f1 = {nb_macro_f1:.3f}   balanced acc = {nb_balanced:.3f}")
        print(f"  {'hmm + viterbi':22}  macro f1 = {hmm_macro_f1:.3f}   balanced acc = {hmm_balanced:.3f}")

        fig, ax = plt.subplots(figsize=(5.2, 4.2))
        sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", xticklabels=["CDR 0", "CDR 0.5", "CDR 1"], yticklabels=["CDR 0", "CDR 0.5", "CDR 1"], cbar=False, ax=ax, annot_kws={"fontsize": 12})
        ax.set_xlabel("predicted cdr")
        ax.set_ylabel("true cdr")
        ax.set_title(f"{name}: HMM confusion matrix (held-out patients)")
        plt.tight_layout()
        plt.show()

    return {
        "name": name,
        "n_features": len(feature_cols),
        "n_visits_train": int(len(y_train)),
        "n_visits_test": int(len(y_test)),
        "train_log_likelihood": float(best_log_lik),
        "state_to_cdr": state_to_cdr,
        "train_agreement": train_agreement,
        "confusion_matrix": confusion,
        "hmm_macro_f1": hmm_macro_f1,
        "hmm_balanced_accuracy": hmm_balanced,
        "majority_macro_f1": majority_macro_f1,
        "majority_balanced_accuracy": majority_balanced,
        "gnb_macro_f1": nb_macro_f1,
        "gnb_balanced_accuracy": nb_balanced,
        "model": best_model,}
